# ODI to Databricks Migration: `insert_trg_loc.sql`

**Converted On:** 2024-07-30

**Description:** This notebook converts an ODI SQL script that inserts data into the `TRG_LOC` table from the `LOCATIONS` source table.
The original Oracle `/*+ APPEND PARALLEL */` hint has been removed, and schema names have been adjusted to Databricks Delta Lake standards. Target table DDL has been inferred.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Target Table Setup

In [ ]:
%sql
-- SCEN_TASK_NO {10}, {20}, {30}: Initialize Target Table
-- Inferred DDL for target table HR.TRG_LOC
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
  LOCATION_ID     BIGINT,
  STREET_ADDRESS  STRING,
  POSTAL_CODE     STRING,
  CITY            STRING,
  STATE_PROVINCE  STRING,
  COUNTRY_ID      STRING
) USING DELTA;

## Insert Data

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Insert into TRG_LOC
-- Original Oracle hint /*+ APPEND PARALLEL */ has been removed.
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID,
  STREET_ADDRESS,
  POSTAL_CODE,
  CITY,
  STATE_PROVINCE,
  COUNTRY_ID
)
SELECT
  LOCATIONS.LOCATION_ID,
  LOCATIONS.STREET_ADDRESS,
  LOCATIONS.POSTAL_CODE,
  LOCATIONS.CITY,
  LOCATIONS.STATE_PROVINCE,
  LOCATIONS.COUNTRY_ID
FROM
  workspace.hr.locations AS LOCATIONS;

In [ ]:
%sql
SELECT COUNT(*) AS records_inserted FROM workspace.hr.trg_loc;

## Optimize Target Table

In [ ]:
%sql
-- Optimize and ZORDER the target table for improved query performance.
-- ZORDER by LOCATION_ID as it is likely a primary key or frequently used filter column.
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_loc ZORDER BY (LOCATION_ID);

## Cleanup

No temporary staging or flow tables were created in this process, so no explicit cleanup is required.

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS final_record_count FROM workspace.hr.trg_loc;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** Original Oracle schema `HR` has been mapped to `workspace.hr`. Table names `TRG_LOC` and `LOCATIONS` have been converted to lowercase (`trg_loc`, `locations`).
2.  **Oracle Hints Removal:** The `/*+ APPEND PARALLEL */` hint specific to Oracle has been removed as it is not applicable in Databricks Delta Lake.
3.  **DDL Inference:** The `CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc` statement was inferred based on typical column types for the given column names in an HR schema context. You may need to verify and adjust the data types and nullability constraints against your actual source system DDL.
4.  **No ODI Parameters:** This specific SQL snippet did not contain any `#GLOBAL` or `#SCHEMA` parameters. However, the boilerplate for `dbutils.widgets` is included as per the standard migration template.
5.  **ZORDER Column Selection:** `LOCATION_ID` was chosen for ZORDER optimization based on the assumption it's a primary key or frequently queried column. Review if other columns or combinations would be more beneficial for your query patterns.